# Exam Project - Introduction to Social Data Science
August 28, 2024

## Project: Forecasting Vote Counts for Danish Borgerforslag

## Group 7:
- Oliver Nyrop Weeks (vsn684)
- Sofus Galavits Møller (qvc730)
- Victor V. Kristensen (gcp458)
- Jonas T. Schmidt (mcp656)

In [1]:
# import modules
import pandas as pd                                                         # We use pandas for datahandling 
import numpy as np
from nltk . stem . snowball import DanishStemmer
import lemmy                                                                # For lemmatization

# import classes
from afinn import Afinn
import nltk
#nltk.download('stopwords')
from nltk.corpus import stopwords                                           # For stopwords in Danish
from sklearn.feature_extraction.text import CountVectorizer                 # Used for topic tagging with LDA
from sklearn.decomposition import LatentDirichletAllocation                 # Used for topic tagging with LDA

Load the dataset:

In [2]:
# Load the dataset from the CSV file
loaded_df = pd.read_csv("data/df_after_variable.csv")

# Display the first few rows of the loaded DataFrame
loaded_df

,Title,Start Date,Votes,Body Text,Num Coauthors,Days from start to scrape,Days from start to end,Lifetime,Month_2,Month_3,...,Title Uppercase Count,Body Text Uppercase Count,Title Lowercase Count,Body Text Lowercase Count,Body Text LIX Score,Distance to parliament km,Title All Upper,Title has Stop,Sentiment_neutral,Sentiment_positive
0,Afskaf inklusionsloven,2024-08-15,39,For at forbedre undervisningen og sikre optima...,3,3,180,3,False,False,...,1,18,20,2556,59.349155,278.896623,False,False,False,True
1,Fartbøder forhindrer i at opnå Dansk statsborg...,2024-08-06,82,Forslaget drejer sig om regler for fartbøder v...,3,12,180,12,False,False,...,2,17,44,999,40.803930,355.768312,False,False,False,True
2,Forbyd dressurridning som konkurrencesport,2024-08-06,245,Dressurridning som konkurrencesport er en spor...,3,12,180,12,False,False,...,1,40,38,1864,40.113388,13.285377,False,False,False,False
3,Lavere skat og fjernelse af minimumsalder på p...,2024-08-06,338,Vi stiller et forslag om ændring af reglerne f...,3,12,180,12,False,False,...,1,14,87,1429,50.907995,291.079559,False,False,False,False
4,Bloddonorpligt. Som værnepligt,2024-07-26,41,Jeg forslår at der etableres en pligt til at a...,3,23,180,23,False,False,...,2,11,25,544,30.552941,338.944018,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1842,Opæhvelse af begrænsning på 45 knallerter,2018-01-30,189,Jeg stiller hermed forslag om at ophævelsen af...,3,2392,180,180,False,False,...,1,6,33,426,43.400000,110.013938,False,False,False,False
1843,Automatisk førtidspension til personer der har...,2018-01-30,66,Forslaget er at man automatisk giver førtidspe...,4,2392,180,180,False,False,...,1,6,108,944,53.763418,390.978286,False,False,False,False
1844,POLITISK FORSLAG TIL HØRING I FOLKETINGET OM Æ...,2018-01-30,597,"Lovforslaget går i sin enkelhed ud på, at sikr...",3,2392,180,180,False,False,...,88,13,0,1385,33.292443,149.484204,True,False,False,True
1845,"Statsborgerskab til unge mennesker, som er fød...",2018-01-30,6263,Folketinget pålægger regeringen at genindføre ...,3,2392,180,180,False,False,...,2,46,88,3886,49.983954,46.247679,False,False,False,True


In [ ]:
# Danish stopwords. This is a small list, enough?
stop = stopwords.words('danish')

# Stemmer in Danish
stemmer = DanishStemmer()

# Lemmatizer in Danish
lemmatizer = lemmy.load("da")

# Create an empty list to store the results
body_tokens = []
body_tokens_stem = []
body_tokens_lem = []

# Loop through each title in the DataFrame column "Body Text"
for document in loaded_df["Body Text"]:

    # Split the title into individual words
    tokens = document.split()

    # Removing stopwords to focus on more meaningful words
    tokens_stop = [i for i in tokens if i not in stop]

    # Stemming: Basicly removing suffixes
    tokens_stemmed = [stemmer.stem(token) for token in tokens_stop]  # Apply stemming to each token individually

    # Apply lemmatization
    tokens_lem = [lemmatizer.lemmatize("", token)[0].lower() for token in tokens_stop]  # Assuming the first lemma is desired

    # Convert the lists of tokens back to a single string (sentence)
    sentence_stemmed = " ".join(tokens_stemmed)
    sentence_lemmatized = " ".join(tokens_lem)
    sentence_tokens = " ".join(tokens_stop)

    # Append the results
    body_tokens_stem.append(sentence_stemmed)
    body_tokens_lem.append(sentence_lemmatized)
    body_tokens.append(sentence_tokens)

# Print the first few rows to verify
print("Original Tokens:", body_tokens[0])
print("Stemmed Tokens:", body_tokens_stem[0])
print("Lemmatized Tokens:", body_tokens_lem[0])


In [ ]:
# Set min_df to 3 (meaning that terms apearing 3 times or less will be removed). We set max_df to 0.1 (meaning that terms apearing in 10% or more documents will be removed)
count = CountVectorizer(min_df=3, max_df=0.1, max_features=50000)
bag = count.fit_transform(body_tokens_lem) # Fit our bag-of-words (given above specifications) and form a bag.

# Define the unsupervised machine learning model with varying components
lda = LatentDirichletAllocation(n_components=10, random_state=123)
borgerforslag_topics = lda.fit_transform(bag) # Borgerforslag_topics contain the topic distribution of each document: Each row represents a document, and each column represents a topic. The values in this matrix represent the probability that a given topic contributes to the document.

n_top_words = 10
word_names = count.get_feature_names_out()
for topic_idx, topic in enumerate(lda.components_): #lda.components_ stores a matrix containing the word importance for each topic
    print("Topic %d:" % (topic_idx + 1))
    print(" ".join([word_names[i]
    for i in topic.argsort()\
        [:-n_top_words - 1:-1]]))
    

In [ ]:
import numpy as np
import pandas as pd

# Assuming borgerforslag_topics is an array with the topic distributions for each document
# and loaded_df is the DataFrame that has been loaded with the dataset

# Assign the most relevant topics to each document
doc_labels = []
for i, topic_dist in enumerate(borgerforslag_topics):
    # Sort topics by probability in descending order
    sorted_topics = np.argsort(topic_dist)[::-1] + 1  # +1 to make topic index human-readable
    
    # Determine the number of relevant topics based on a threshold or top N selection
    top_n = 3  # For example, we want to assign up to 3 topics
    top_topics = sorted_topics[:top_n]
    
    # Filter topics based on a probability threshold (optional)
    threshold = 0.35  # Only include topics with a probability above 0.35
    relevant_topics = [topic for topic in top_topics if topic_dist[topic - 1] > threshold]
    
    # Assign the relevant topics to the document
    for topic in relevant_topics:
        doc_labels.append((i, topic))


# Create a DataFrame from the doc_labels with the index as document index
doc_labels_df = pd.DataFrame(doc_labels, columns=['index', 'topic'])

# Create dummies for the topics and sum them by document index to avoid duplicates
topic_dummies = pd.get_dummies(doc_labels_df['topic'], prefix="topic")
topic_dummies = doc_labels_df.join(topic_dummies).groupby('index').sum().reset_index()

# Ensure the dummy dataframe includes all document indices
# Merge with a DataFrame containing all possible document indices to ensure all documents are included
all_indices_df = pd.DataFrame({'index': range(len(loaded_df))})
topic_dummies = pd.merge(all_indices_df, topic_dummies, on='index', how='left').fillna(0)

# Convert the dummy values to integers (removing any floating-point representation)
topic_dummies.iloc[:, 1:] = topic_dummies.iloc[:, 1:].astype(int)

# Concatenate the topic dummies with the original DataFrame
df_xy = pd.concat([loaded_df, topic_dummies.drop(columns=['index'])], axis=1)

# Display the resulting DataFrame
df_xy


In [ ]:
# Specify the path where you want to save the CSV file
output_path = 'Data/df_after_categorization.csv'

# Save the DataFrame to the CSV file
df_xy.to_csv(output_path, index=False)

# Print the output path to confirm where the file was saved
print(f"DataFrame saved to: {output_path}")